# JAX conditioning, filtering and veto boundaries

Fixing input bytes, dtype, grids and options distinguishes numerical changes from different calculations. Each original-implementation control below asserts dtype, shape, metadata and bytes against an ordinary CPU run. Default JAX remains selected except at named opt-in validation boundaries.

Synthetic signals illustrate numerical mechanisms; they do not generate astrophysical templates or qualify a complete search. Recorded outputs apply to the displayed libraries and device. Install PyCBC, JAX and Jupyter in the same environment.

For a **self-contained NumPy/JAX demonstration** without PyCBC or LAL, run
[filtering rounding](jax_filtering_rounding.ipynb). That notebook illustrates arithmetic
with generated inputs. This notebook checks the actual PyCBC implementations
and original validation routes. Results apply to each notebook's recorded
libraries and device.


In [1]:
import platform
import warnings
warnings.filterwarnings('ignore', 'Wswiglal-redir-stdio')
import numpy as np
import scipy
from scipy.signal import firwin as scipy_firwin
import jax
import jax.numpy as jnp
import jaxlib
import lal
import lalsimulation
import pycbc
from pycbc.scheme import CPUScheme, JAXScheme
from pycbc.types import Array, TimeSeries, FrequencySeries
from pycbc.fft import backend_cpu
from pycbc.filter import sigmasq, matched_filter_core
from pycbc.filter.resample import highpass, lowpass, lfilter, fir_zero_filter, resample_to_delta_t
from pycbc.filter.autocorrelation import calculate_acf
from pycbc.vetoes.chisq import (shift_sum, power_chisq_bins, chisq_accum_bin,
                               power_chisq_at_points_from_precomputed)

DEVICE = 'cpu'  # Repeat with 'cuda:0' on an available NVIDIA GPU.
with JAXScheme(DEVICE) as context:
    metadata = {'Python': platform.python_version(),
                'CPU architecture': platform.machine(), 'PyCBC': pycbc.__version__,
                'NumPy': np.__version__, 'SciPy': scipy.__version__,
                'JAX': jax.__version__, 'jaxlib': jaxlib.__version__,
                'LAL': lal.__version__, 'LALSimulation': lalsimulation.__version__,
                'CPU FFT backend': backend_cpu.get_backend().__name__,
                'JAX platform': context.jax_device.platform,
                'JAX device kind': context.jax_device.device_kind,
                'JAX x64': jax.config.jax_enable_x64}
assert metadata['JAX x64']
for name, value in metadata.items():
    print(f'{name}: {value}')
rng = np.random.default_rng(1729)
samples = rng.normal(size=256)
template_values = (rng.normal(size=129) + 1j*rng.normal(size=129)).astype(np.complex64)
data_values = (rng.normal(size=129) + 1j*rng.normal(size=129)).astype(np.complex64)
psd_values = (1 + .1*np.arange(129)).astype(np.float32)
correlation_values = (rng.normal(size=256) + 1j*rng.normal(size=256)).astype(np.complex64)
points = np.array([0, 8, 63, 127], dtype=np.int64)
bins = np.array([1, 17, 33, 65], dtype=np.uint32)


Python: 3.12.7
CPU architecture: arm64
PyCBC: 0.0a9304
NumPy: 2.5.2
SciPy: 1.16.3
JAX: 0.11.1
jaxlib: 0.11.1
LAL: 7.7.1
LALSimulation: 6.2.1
CPU FFT backend: pycbc.fft.fftw
JAX platform: cpu
JAX device kind: cpu
JAX x64: True


In [2]:
def snapshot(value):
    if isinstance(value, tuple):
        return tuple(snapshot(item) for item in value)
    values = np.array(value.numpy() if hasattr(value, 'numpy') else value, copy=True)
    result = {'values': values}
    if isinstance(value, (TimeSeries, FrequencySeries)):
        epoch = value.start_time if isinstance(value, TimeSeries) else value.epoch
        result['epoch'] = None if epoch is None else str(epoch)
        for attribute in ('delta_t', 'delta_f', 'corrupted_samples'):
            if hasattr(value, attribute):
                result[attribute] = getattr(value, attribute)
    return result


def assert_exact(actual, expected):
    if isinstance(expected, tuple):
        assert len(actual) == len(expected)
        for a, b in zip(actual, expected):
            assert_exact(a, b)
        return
    a, b = actual['values'], expected['values']
    assert a.shape == b.shape and a.dtype == b.dtype
    assert a.tobytes() == b.tobytes()
    assert {k:v for k,v in actual.items() if k != 'values'} == {
        k:v for k,v in expected.items() if k != 'values'}


def report(name, cpu, candidate):
    if isinstance(cpu, tuple):
        for index, (a, b) in enumerate(zip(cpu, candidate)):
            report(f'{name}/output{index}', a, b)
        return
    a, b = cpu['values'], candidate['values']
    if a.shape != b.shape or a.size == 0 or a.dtype.kind not in 'fc':
        print(f'{name}: CPU/JAX values={a}/{b}; shapes={a.shape}/{b.shape}')
        return
    finite = np.isfinite(a) & np.isfinite(b)
    error = float(np.max(np.abs(a[finite] - b[finite])))
    scale = max(float(np.max(np.abs(a[finite]))), np.finfo(a.dtype).tiny)
    print(f'{name}: CPU/JAX dtype={a.dtype}/{b.dtype}; max abs={error:.6g}; '
          f'scaled max={error/scale:.6g}; '
          f'value mismatches={np.count_nonzero(a != b)}/{a.size}')


def evaluate(operation, values=None, *, cpu=False, reference=(), mode='cpu-compatible',
             **options):
    context = CPUScheme() if cpu else JAXScheme(
        DEVICE, reference_operations=reference, chisq_mode=mode)
    with context:
        if operation in ('highpass', 'lowpass', 'lfilter', 'fir_zero_filter', 'resample', 'autocorrelation'):
            spacing = options.get('delta_t', 1/1024) if operation == 'autocorrelation' else 1/1024
            source = TimeSeries(values.copy(), delta_t=spacing, epoch=0)
            if operation in ('lfilter', 'fir_zero_filter'):
                fn = lfilter if operation == 'lfilter' else fir_zero_filter
                result = fn(options.pop('coefficients'), source, **options)
            elif operation == 'resample':
                result = resample_to_delta_t(source, **options)
            elif operation == 'autocorrelation':
                result = calculate_acf(source, **options)
            else:
                result = (highpass if operation == 'highpass' else lowpass)(source, **options)
        elif operation in ('sigmasq', 'matched_filter_core', 'power_chisq_bins'):
            template = FrequencySeries(template_values.copy(), delta_f=4, epoch=0)
            data = FrequencySeries(data_values.copy(), delta_f=4, epoch=0)
            psd = FrequencySeries(psd_values.copy(), delta_f=4, epoch=0)
            if operation == 'sigmasq':
                result = sigmasq(template, psd=psd, **options)
            elif operation == 'power_chisq_bins':
                result = power_chisq_bins(template, psd=psd, **options)
            else:
                result = matched_filter_core(template, data, psd=psd, **options)
        elif operation == 'shift_sum':
            result = shift_sum(Array(correlation_values.copy()), points.copy(), bins.copy())
        elif operation == 'power_chisq_at_points':
            corr = FrequencySeries(correlation_values.copy(), delta_f=.5)
            snrv = np.array([1+2j, 2-3j, 4+1j, -2+4j], dtype=np.complex64)
            result = power_chisq_at_points_from_precomputed(
                corr, snrv, .123456789, bins.copy(), points.copy())
        elif operation == 'chisq_accum_bin':
            result = Array(np.linspace(.1, .3, 256, dtype=np.float32))
            chisq_accum_bin(result, Array(correlation_values.copy()))
        else:
            raise ValueError(operation)
        return snapshot(result)


checks = {}
def compare(name, operation, values=None, reference=None, **options):
    reference = (operation,) if reference is None else reference
    cpu = evaluate(operation, values, cpu=True, **options)
    default = evaluate(operation, values, **options)
    native = evaluate(operation, values, reference=reference, **options)
    assert_exact(native, cpu)
    report(name, cpu, default)
    print('  original implementation dtype/shape/metadata/bytes: exact')
    checks[name] = (cpu, default, native)
    return cpu, default, native


## On-device Butterworth filters and original LAL controls

These comparisons hold the filter order, cutoff and attenuation fixed. Default JAX composes affine state updates on device. The opt-in stage route executes the original LAL calculation, including its coefficients, histories and sample order.

In [3]:
for dtype in (np.float32, np.float64):
    _ = compare(dtype.__name__ + '/highpass', 'highpass', samples.astype(dtype),
                frequency=30, filter_order=4, attenuation=.1)
_ = compare('float32/lowpass', 'lowpass', samples.astype(np.float32),
            frequency=100, filter_order=4, attenuation=.1)

# An additive affine recurrence isolates finite-precision composition order.
with JAXScheme(DEVICE):
    terms = jnp.array([1e16, 1., -1e16, 1.], dtype=jnp.float64)
    sequential = jax.lax.scan(lambda total, x: (total+x, total+x),
                             jnp.float64(0), terms)[1]
    parallel = jax.lax.associative_scan(jnp.add, terms)
print('Additive recurrence sequential/parallel final values:',
      float(sequential[-1]), float(parallel[-1]))
assert float(sequential[-1]) == 1 and float(parallel[-1]) == 0


float32/highpass: CPU/JAX dtype=float32/float32; max abs=0; scaled max=0; value mismatches=0/256
  original implementation dtype/shape/metadata/bytes: exact


float64/highpass: CPU/JAX dtype=float64/float64; max abs=2.41696e-13; scaled max=9.44351e-14; value mismatches=254/256
  original implementation dtype/shape/metadata/bytes: exact


float32/lowpass: CPU/JAX dtype=float32/float32; max abs=0; scaled max=0; value mismatches=0/256
  original implementation dtype/shape/metadata/bytes: exact
Additive recurrence sequential/parallel final values: 1.0 0.0


## FIR design, normalization and resampling

One common coefficient array isolates filtering from coefficient design. Short records retain the original promoted causal precision; long records retain its circular FFT convention. Resampling includes coefficient design before decimation.

The compensated normalization control retains the two lost unit terms in a cancellation sum. That sum alone does not make window/sinc calculations match another numerical library.

In [4]:
coefficients = np.array([.2, .3, .5], dtype=np.float64)
_ = compare('short float32 FIR', 'lfilter', samples[:64].astype(np.float32),
            coefficients=coefficients)
cpu_fir, _, _ = compare('circular float64 FIR', 'lfilter', samples.astype(np.float64),
            coefficients=coefficients)
_ = compare('zero-phase float32 FIR', 'fir_zero_filter', samples.astype(np.float32),
            coefficients=coefficients)
_ = compare('float32 LDAS resampling', 'resample', samples.astype(np.float32),
            delta_t=1/512, method='ldas')

controlled_fir = evaluate('lfilter', samples.astype(np.float64),
                          reference=('fft', 'ifft'), coefficients=coefficients)
report('circular FIR/original transforms', cpu_fir, controlled_fir)

from pycbc.filter.resample_jax import firwin, _compensated_pairwise_sum
cpu_coefficients = scipy_firwin(41, .5, window=('kaiser', 5.))
with JAXScheme(DEVICE):
    jax_coefficients = np.asarray(firwin(41, .5, window=('kaiser', 5.)))
    compensated = float(_compensated_pairwise_sum(terms))
with JAXScheme(DEVICE, reference_operations=('firwin',)):
    native_coefficients = np.asarray(firwin(41, .5, window=('kaiser', 5.)))
assert native_coefficients.dtype == cpu_coefficients.dtype
assert native_coefficients.tobytes() == cpu_coefficients.tobytes()
print('Original FIR coefficients dtype/shape/bytes: exact')
print('41-tap Kaiser coefficient maximum error:',
      np.max(np.abs(cpu_coefficients-jax_coefficients)))
print('Compensated cancellation sum:', compensated, '; exact real sum: 2')
assert compensated == 2


short float32 FIR: CPU/JAX dtype=float64/float64; max abs=0; scaled max=0; value mismatches=0/64
  original implementation dtype/shape/metadata/bytes: exact


circular float64 FIR: CPU/JAX dtype=float64/float64; max abs=8.88178e-16; scaled max=4.69547e-16; value mismatches=220/256
  original implementation dtype/shape/metadata/bytes: exact


zero-phase float32 FIR: CPU/JAX dtype=float32/float32; max abs=3.42727e-07; scaled max=1.81187e-07; value mismatches=199/256
  original implementation dtype/shape/metadata/bytes: exact


float32 LDAS resampling: CPU/JAX dtype=float32/float32; max abs=3.57628e-07; scaled max=1.9007e-07; value mismatches=88/128
  original implementation dtype/shape/metadata/bytes: exact
circular FIR/original transforms: CPU/JAX dtype=float64/float64; max abs=4.44089e-16; scaled max=2.34774e-16; value mismatches=169/256
Original FIR coefficients dtype/shape/bytes: exact
41-tap Kaiser coefficient maximum error: 5.551115123125783e-17
Compensated cancellation sum: 2.0 ; exact real sum: 2


## Matched-filter stage isolation

The PSD-weighted norm uses the `weighted_inner` boundary. Selecting it together with `correlate`, `divide` and `ifft` restores the standard filter's original numerical components for this fixture. Original correlation and inverse transform alone leave JAX PSD division in place. A supplied CPU normalization isolates correlation, division and transform stages.

In [5]:
cutoffs = dict(low_frequency_cutoff=20, high_frequency_cutoff=200)
cpu_sigma, _, _ = compare('complex64 template norm', 'sigmasq',
                          reference=('weighted_inner',), **cutoffs)
cpu_filter, _, _ = compare('complex64 matched filter', 'matched_filter_core',
                           reference=('weighted_inner', 'correlate', 'divide', 'ifft'), **cutoffs)
fixed_norm = float(cpu_sigma['values'])
for selected in ((), ('correlate',), ('ifft',), ('correlate', 'ifft'),
                 ('correlate', 'divide', 'ifft')):
    result = evaluate('matched_filter_core', reference=selected, h_norm=fixed_norm, **cutoffs)
    report('fixed norm/' + (','.join(selected) or 'JAX'), cpu_filter, result)

# Hold the division input fixed and compare explicit rounding sequences.
division_rng = np.random.default_rng(334)
dividend = (division_rng.normal(size=65)+1j*division_rng.normal(size=65)).astype(np.complex64)
denominator = division_rng.uniform(.5, 2., 65).astype(np.float32)
with CPUScheme():
    cpu_division = snapshot(Array(dividend.copy()) / Array(denominator.copy()))
with JAXScheme(DEVICE):
    default_division = snapshot(Array(dividend.copy()) / Array(denominator.copy()))
with JAXScheme(DEVICE, reference_operations=('divide',)):
    native_division = snapshot(Array(dividend.copy()) / Array(denominator.copy()))
assert_exact(native_division, cpu_division)
component_division = np.empty_like(dividend)
component_division.real = dividend.real / denominator
component_division.imag = dividend.imag / denominator
reciprocal_multiplication = dividend * (np.float32(1) / denominator)
assert cpu_division['values'].tobytes() == reciprocal_multiplication.tobytes()
assert default_division['values'].tobytes() == component_division.tobytes()
report('fixed complex64/float32 division', cpu_division, default_division)
print('NumPy bytes equal rounded-reciprocal multiplication; JAX bytes equal component division.')
print('  original division dtype/shape/bytes: exact')
checks['PSD division'] = (cpu_division, default_division, native_division)


complex64 template norm: CPU/JAX dtype=float64/float64; max abs=3.30089e-07; scaled max=6.9512e-10; value mismatches=1/1
  original implementation dtype/shape/metadata/bytes: exact
complex64 matched filter/output0: CPU/JAX dtype=complex64/complex64; max abs=1.92219e-06; scaled max=1.57134e-07; value mismatches=244/256
complex64 matched filter/output1: CPU/JAX dtype=complex64/complex64; max abs=2.38419e-07; scaled max=5.83577e-08; value mismatches=26/256
complex64 matched filter/output2: CPU/JAX dtype=float64/float64; max abs=2.55191e-10; scaled max=3.4756e-10; value mismatches=1/1
  original implementation dtype/shape/metadata/bytes: exact
fixed norm/JAX/output0: CPU/JAX dtype=complex64/complex64; max abs=1.92219e-06; scaled max=1.57134e-07; value mismatches=244/256
fixed norm/JAX/output1: CPU/JAX dtype=complex64/complex64; max abs=2.38419e-07; scaled max=5.83577e-08; value mismatches=26/256
fixed norm/JAX/output2: CPU/JAX dtype=float64/float64; max abs=0; scaled max=0; value mismatche

## Autocorrelation and point chi-square

Autocorrelation adds centering and normalization to transform differences. Its mean and variance can be routed independently to NumPy while transforms remain JAX; the combined reduction/transform/correlation/division controls are checked exactly. Point chi-square holds the frequency-bin edges and selected samples fixed; default recurrence-based JAX and independent direct phases are compared with the original compiled CPU implementation.

The two phase controls separate the CPU seed's pi precision and conversion of sample indices to float32. They do not identify every residual in a complete veto value.

In [6]:
acf_values = np.random.default_rng(123).normal(size=127).astype(np.float32)
cpu_acf, default_acf, _ = compare('float32 autocorrelation', 'autocorrelation', acf_values,
            unbiased=True, delta_t=.125)
fixed_centered = acf_values - np.mean(acf_values)
with JAXScheme(DEVICE):
    device_mean = np.asarray(jnp.mean(jnp.asarray(acf_values)))
    device_centered = np.asarray(jnp.asarray(acf_values)-device_mean)
    device_variance = float(jnp.var(jnp.asarray(fixed_centered)))
    common_power = float(jnp.sum(jnp.asarray(fixed_centered, dtype=jnp.float64)**2))
assert device_mean.tobytes() == np.mean(acf_values).tobytes()
assert device_centered.tobytes() == fixed_centered.tobytes()
assert common_power == np.sum(fixed_centered.astype(np.float64)**2)
print('Mean/centered bytes and double-precision power sum: equal')
print('Fixed centered-input variance CPU/JAX:', np.var(fixed_centered), device_variance)
lag_zero_cpu = .125*common_power/(len(acf_values)*float(np.var(fixed_centered)))
lag_zero_jax = .125*common_power/(len(acf_values)*device_variance)
print('Variance-only zero-lag controls CPU/JAX:', lag_zero_cpu, lag_zero_jax)
print('Actual zero-lag ACF CPU/JAX:', cpu_acf['values'][0], default_acf['values'][0])
for selected in (('autocorrelation_mean',), ('autocorrelation_variance',)):
    partial = evaluate('autocorrelation', acf_values, reference=selected,
                       unbiased=True, delta_t=.125)
    report('ACF/' + selected[0], cpu_acf, partial)
acf_boundaries = ('autocorrelation_mean', 'autocorrelation_variance',
                  'fft', 'ifft', 'correlate', 'divide')
controlled_acf = evaluate('autocorrelation', acf_values, reference=acf_boundaries,
                          unbiased=True, delta_t=.125)
assert_exact(controlled_acf, cpu_acf)
report('ACF/original component controls', cpu_acf, controlled_acf)
print('  original ACF component dtype/shape/metadata/bytes: exact')
checks['granular autocorrelation'] = (cpu_acf, default_acf, controlled_acf)
cpu_points, _, _ = compare('complex64 point chi-square', 'shift_sum')
direct = evaluate('shift_sum', mode='direct-phase')
report('independent direct phase', cpu_points, direct)
_ = compare('complex64 full point chi-square', 'power_chisq_at_points')
_ = compare('float32 bin accumulation', 'chisq_accum_bin')
_ = compare('template-power bin edges', 'power_chisq_bins', num_bins=4, **cutoffs)

n_time, frequency_bin, sample = 65536, 32000, 65535
original_angle = 2*3.141592653*sample*frequency_bin/n_time
full_pi_angle = 2*np.pi*sample*frequency_bin/n_time
print('Pi-only seed phase difference:',
      abs(np.exp(1j*original_angle)-np.exp(1j*full_pi_angle)))

n_time, frequency_bin, sample = 2**25, 16384, 2**24+1
converted = int(np.float32(sample))
cycles = (frequency_bin*sample) % n_time
converted_cycles = (frequency_bin*converted) % n_time
print('Index conversion original/float32:', sample, converted)
print('Integer phase cycles original/after conversion:', cycles, converted_cycles)
assert converted != sample and cycles != converted_cycles


float32 autocorrelation: CPU/JAX dtype=float64/float64; max abs=8.51367e-09; scaled max=6.81093e-08; value mismatches=127/127
  original implementation dtype/shape/metadata/bytes: exact
Mean/centered bytes and double-precision power sum: equal
Fixed centered-input variance CPU/JAX: 0.8751317 0.8751317858695984
Variance-only zero-lag controls CPU/JAX: 0.12500000523536803 0.12499999672170069
Actual zero-lag ACF CPU/JAX: 0.12500000523536806 0.12499999672170069
ACF/autocorrelation_mean: CPU/JAX dtype=float64/float64; max abs=8.51367e-09; scaled max=6.81093e-08; value mismatches=127/127
ACF/autocorrelation_variance: CPU/JAX dtype=float64/float64; max abs=1.90305e-16; scaled max=1.52244e-15; value mismatches=114/127
ACF/original component controls: CPU/JAX dtype=float64/float64; max abs=0; scaled max=0; value mismatches=0/127
  original ACF component dtype/shape/metadata/bytes: exact
complex64 point chi-square: CPU/JAX dtype=float32/float32; max abs=1.52588e-05; scaled max=6.02807e-08; value

complex64 full point chi-square: CPU/JAX dtype=float32/float32; max abs=9.53674e-07; scaled max=8.46249e-08; value mismatches=2/4
  original implementation dtype/shape/metadata/bytes: exact
float32 bin accumulation: CPU/JAX dtype=float32/float32; max abs=0; scaled max=0; value mismatches=0/256
  original implementation dtype/shape/metadata/bytes: exact


template-power bin edges: CPU/JAX values=[ 5  9 13 29 50]/[ 5  9 13 29 50]; shapes=(5,)/(5,)
  original implementation dtype/shape/metadata/bytes: exact
Pi-only seed phase difference: 3.774618380479396e-05
Index conversion original/float32: 16777217 16777216
Integer phase cycles original/after conversion: 16384 0


## A threshold crossing caused by one rounding step

This fixed complex64 sample lies at a magnitude threshold. The original CPU cluster class and default JAX use different floating-point product/addition grouping. A one-step difference can therefore change a discrete trigger decision. The native cluster selector restores both the original values and indices.

In [7]:
from pycbc.events.threshold_cpu import CPUThresholdCluster
from pycbc.events.threshold_jax import JAXThresholdCluster

boundary_sample = np.array([complex(-.7931225, -.13891791)], dtype=np.complex64)
threshold_value = .8051965236663818
with CPUScheme():
    cpu_cluster = snapshot(CPUThresholdCluster(Array(boundary_sample.copy()))
                           .threshold_and_cluster(threshold_value, 1))
with JAXScheme(DEVICE):
    default_cluster = snapshot(JAXThresholdCluster(Array(boundary_sample.copy()))
                               .threshold_and_cluster(threshold_value, 1))
    fused_power = np.asarray(jax.jit(lambda z: z.real*z.real + z.imag*z.imag)(
        jnp.asarray(boundary_sample)))
with JAXScheme(DEVICE, reference_operations=('threshold_cluster',)):
    native_cluster = snapshot(JAXThresholdCluster(Array(boundary_sample.copy()))
                              .threshold_and_cluster(threshold_value, 1))
assert_exact(native_cluster, cpu_cluster)
separate_power = boundary_sample.real**2 + boundary_sample.imag**2
retained_product = np.float32(boundary_sample.real.astype(np.float64)*boundary_sample.real + boundary_sample.imag**2)
assert retained_product.tobytes() == fused_power.tobytes()
print('One-product retained-precision control:', retained_product)
threshold_f32 = np.float32(threshold_value)
print('Separate/JAX fused power:', separate_power, fused_power)
print('Float32 squared threshold:', threshold_f32*threshold_f32)
print('CPU/JAX selected indices:', cpu_cluster[1]['values'],
      default_cluster[1]['values'])
print('Original cluster values/indices/dtypes/bytes: exact')
checks['threshold cluster'] = (cpu_cluster, default_cluster, native_cluster)


One-product retained-precision control: [0.6483414]
Separate/JAX fused power: [0.6483415] [0.6483414]
Float32 squared threshold: 0.6483414
CPU/JAX selected indices: [0] []
Original cluster values/indices/dtypes/bytes: exact


## Sine-Gaussian veto boundaries

The internal sine-Gaussian tiles support the veto calculation. Original tile construction, frequency-domain time shifting and the complete veto have independent controls. The complete comparison fixes bin edges, template metadata, activation inputs and sample indices; its nontrivial tile values are evaluated below. Selected tile/shift/reduction controls retain the remaining JAX arithmetic.

In [8]:
from types import SimpleNamespace
from pycbc.vetoes.sgchisq import SingleDetSGChisq
from pycbc.vetoes.sgchisq_jax import _sine_gaussian_basis
from pycbc.waveform.sinegauss import fd_sine_gaussian
from pycbc.waveform.utils import apply_fseries_time_shift

basis_args = (1., 8., 100., 20., 500., .25)
with CPUScheme():
    cpu_basis = snapshot(fd_sine_gaussian(*basis_args))
with JAXScheme(DEVICE):
    default_basis = snapshot(_sine_gaussian_basis(*basis_args))
with JAXScheme(DEVICE, reference_operations=('sg_basis',)):
    native_basis = snapshot(_sine_gaussian_basis(*basis_args))
assert_exact(native_basis, cpu_basis)
report('internal sine-Gaussian basis', cpu_basis, default_basis)
print('  original tile dtype/shape/metadata/bytes: exact')
checks['SG basis'] = (cpu_basis, default_basis, native_basis)

shift_values = correlation_values[:129].copy()
def shifted():
    source = FrequencySeries(shift_values.copy(), delta_f=.5, epoch=12)
    return snapshot(apply_fseries_time_shift(source, .037, kmin=11))
with CPUScheme():
    cpu_shift = shifted()
with JAXScheme(DEVICE):
    default_shift = shifted()
with JAXScheme(DEVICE, reference_operations=('time_shift',)):
    native_shift = shifted()
assert_exact(native_shift, cpu_shift)
report('complex64 frequency-domain time shift', cpu_shift, default_shift)
print('  original shift dtype/shape/metadata/bytes: exact')
checks['time shift'] = (cpu_shift, default_shift, native_shift)

sg_rng = np.random.default_rng(4)
sg_data = (sg_rng.normal(size=1025)+1j*sg_rng.normal(size=1025)).astype(np.complex64)
sg_template = np.ones(1025, dtype=np.complex64)
sg_psd = np.ones(1025, dtype=np.float32)
sg_bins = np.array([10, 80, 120, 160], dtype=np.uint32)
calculator = SingleDetSGChisq.__new__(SingleDetSGChisq)
calculator.do = True
calculator.params = {1: '8-30,12-60'}
calculator.snr_threshold = 0
calculator.cached_chisq_bins = lambda template, psd: sg_bins
sg_args = (np.array([10+1j, 20+2j], dtype=np.complex64), 1.,
           np.ones(2), np.full(2, 6), np.array([1, 41], dtype=np.uint32))
def sg_values():
    template = FrequencySeries(sg_template.copy(), delta_f=1., epoch=0)
    template.f_lower = 10
    template.params = SimpleNamespace(template_hash=1)
    return snapshot(calculator.values(FrequencySeries(sg_data.copy(), delta_f=1.),
                    template, FrequencySeries(sg_psd.copy(), delta_f=1.), *sg_args))
with CPUScheme():
    cpu_sg = sg_values()
with JAXScheme(DEVICE):
    default_sg = sg_values()
with JAXScheme(DEVICE, reference_operations=('sgchisq',)):
    native_sg = sg_values()
assert_exact(native_sg, cpu_sg)
assert np.all(cpu_sg['values'] != 1)
report('complete sine-Gaussian veto', cpu_sg, default_sg)
print('  original complete veto dtype/shape/bytes: exact')
checks['SG veto'] = (cpu_sg, default_sg, native_sg)
with JAXScheme(DEVICE, reference_operations=('sg_basis', 'time_shift', 'weighted_inner', 'sum', 'divide')):
    controlled_sg = sg_values()
assert_exact(controlled_sg, cpu_sg)
report('selected original SG components', cpu_sg, controlled_sg)
print('Active original SG veto values:', cpu_sg['values'])


internal sine-Gaussian basis: CPU/JAX dtype=complex128/complex128; max abs=1.73472e-18; scaled max=1.53736e-16; value mismatches=104/2000
  original tile dtype/shape/metadata/bytes: exact


complex64 frequency-domain time shift: CPU/JAX dtype=complex64/complex64; max abs=1.62142e-06; scaled max=4.97542e-07; value mismatches=110/129
  original shift dtype/shape/metadata/bytes: exact


complete sine-Gaussian veto: CPU/JAX dtype=float64/float64; max abs=1.96503e-06; scaled max=6.20339e-07; value mismatches=2/2
  original complete veto dtype/shape/bytes: exact


selected original SG components: CPU/JAX dtype=float64/float64; max abs=0; scaled max=0; value mismatches=0/2
Active original SG veto values: [3.16767263 1.03437315]


## Validation scope

Original implementation controls are opt-in; other stages remain JAX. Host transfers and isolated CPU process startup make whole-stage routes slower, and JAX tracing/differentiation cannot cross them. Canonical array/FFT controls restore their corresponding original kernels independently.

A numerical difference from the original CPU value remains a failed equality comparison; an alternative mathematical oracle does not replace that reference. These small controls do not qualify a complete search, every configuration or a different device. See [filtering guide](../../docs/jax_filtering.rst) and [numerical differences](../../docs/jax_filtering_numerical_differences.rst).

## Coherent signals amplify preceding rounding

Here the raw SNR comes from the original inverse transform of the **same** correlation used by the point veto. Four equal-width bins give `A = p*S = 55281`, `B = |raw_snr|² = 55112` and `chi_square = 169`, with norm one. Sample zero removes phase evolution: this isolates the sensitivity of subtracting the two large terms.

The default device calculation is measured, and the independent original point control must match the complete original calculation in dtype, shape and bytes. We then advance either the held bin-power sum or one raw SNR component by exactly one float32 representable step. These controlled perturbations demonstrate amplification; they do not claim that every compiler or device produces those perturbations, or assign all remaining kernel bits to fusion or trigonometric functions.

In [9]:
import numpy as np
from pycbc.fft import ifft
from pycbc.scheme import CPUScheme, JAXScheme
from pycbc.types import Array
from pycbc.vetoes.chisq import (shift_sum,
                               power_chisq_at_points_from_precomputed)

COHERENT_DEVICE = 'cpu'  # Repeat with 'cuda:0' on an available NVIDIA GPU.


def coherent_values(value):
    return np.array(value.numpy() if hasattr(value, 'numpy') else value, copy=True)


def coherent_exact(actual, expected):
    assert actual.shape == expected.shape and actual.dtype == expected.dtype
    assert actual.tobytes() == expected.tobytes()


coherent_bins = np.array([1, 17, 33, 49, 65], dtype=np.uint32)
coherent_points = np.array([0], dtype=np.int64)
coherent_corr = np.zeros(256, dtype=np.complex64)
responses = np.array([44.75+41.5j, 38.25+41.5j]*2, dtype=np.complex64)
for start, end, response in zip(coherent_bins[:-1], coherent_bins[1:], responses):
    coherent_corr[start:end] = response / (end-start)

with CPUScheme():
    correlation = Array(coherent_corr.copy())
    snr_buffer = Array(np.zeros_like(coherent_corr))
    ifft(correlation, snr_buffer)
    coherent_peak = snr_buffer.numpy()[coherent_points].copy()
    coherent_sum = shift_sum(correlation, coherent_points, coherent_bins)
    original_point = coherent_values(power_chisq_at_points_from_precomputed(
        correlation, coherent_peak.copy(), 1., coherent_bins, coherent_points))

with JAXScheme(COHERENT_DEVICE) as coherent_context:
    default_point = coherent_values(power_chisq_at_points_from_precomputed(
        Array(coherent_corr.copy()), coherent_peak.copy(), 1.,
        coherent_bins, coherent_points))
    print('Coherent proof device:', coherent_context.jax_device.platform,
          coherent_context.jax_device.device_kind)
with JAXScheme(COHERENT_DEVICE, reference_operations=('power_chisq_at_points',)):
    original_control = coherent_values(power_chisq_at_points_from_precomputed(
        Array(coherent_corr.copy()), coherent_peak.copy(), 1.,
        coherent_bins, coherent_points))
coherent_exact(original_control, original_point)
print('Original/default point:', original_point, default_point,
      '; differing values:', np.count_nonzero(original_point != default_point))

# The batched veto's final NumPy expression, with only its named input changed.
def coherent_final(bin_power_sum, raw_peak):
    return 4*bin_power_sum - (raw_peak.conj()*raw_peak).real


coherent_chisq = coherent_final(coherent_sum, coherent_peak)
coherent_exact(coherent_chisq, original_point)
assert coherent_chisq.tolist() == [169.]
A = 4*coherent_sum
B = (coherent_peak.conj()*coherent_peak).real
sum_one_step = np.nextafter(coherent_sum, np.float32(np.inf))
peak_one_step = coherent_peak.copy()
peak_one_step.real = np.nextafter(peak_one_step.real, np.float32(np.inf))
chisq_sum_step = coherent_final(sum_one_step, coherent_peak)
chisq_peak_step = coherent_final(coherent_sum, peak_one_step)
power_one_step = (peak_one_step.conj()*peak_one_step).real
# Float64 reports exact differences between stored float32 values only.
sum_relative = (sum_one_step.astype(np.float64)-coherent_sum)/coherent_sum
power_relative = (power_one_step.astype(np.float64)-B)/B
sum_amplification = ((chisq_sum_step.astype(np.float64)-coherent_chisq)
                     / coherent_chisq / sum_relative)
power_amplification = ((chisq_peak_step.astype(np.float64)-coherent_chisq)
                       / coherent_chisq / power_relative)
assert (chisq_sum_step-coherent_chisq).tolist() == [0.00390625]
assert (chisq_peak_step-coherent_chisq).tolist() == [-0.00390625]
print('Same-correlation raw SNR:', coherent_peak, '; bin power:', coherent_sum)
print('A, B, A-B:', A, B, coherent_chisq)
print('One-step bin-power/SNR-component chisq:', chisq_sum_step, chisq_peak_step)
print('Relative-error amplification from bin power:', sum_amplification)
print('Relative-error amplification from raw SNR power:', power_amplification)
print('Original point dtype/shape/bytes: exact')


Coherent proof device: cpu cpu


Original/default point: [169.] [169.] ; differing values: 0
Same-correlation raw SNR: [166.+166.j] ; bin power: [13820.25]
A, B, A-B: [55281.] [55112.] [169.]
One-step bin-power/SNR-component chisq: [169.0039] [168.9961]
Relative-error amplification from bin power: [327.10650888]
Relative-error amplification from raw SNR power: [-326.10650888]
Original point dtype/shape/bytes: exact


## Point-sum rounding caused by regrouped additions

Both implementations use the same three-product complex multiplication,
`a = vr*(pr+pi)`, `b = pr*(vi-vr)`, `c = pi*(vr+vi)`. Matching that source
formula does not fix the order in which a compiled loop rounds additions.
In the observed scalar float32 path of a qualified original CPU build,
pairs of frequency updates were grouped as
`R = ((a1-c1)+(a0-c0))+R` and
`I = ((I+b0)+(b1+a0))+a1`; its scalar remainder used
`I = (b+I)+a`. The observed JAX device loop instead updated
`R = R+(a-c)` and `I = I+(a+b)` at each frequency. Native final accumulation
also used `(S+R*R)+I*I`, whereas the device grouped a bin's power first:
`S+(R*R+I*I)`.

The NumPy control below evaluates those groupings with a float32 rounding
boundary at every operation. Six generated correlation samples, arranged in
two three-frequency bins, suffice to produce a difference. Sample zero fixes
all phase seeds and updates to `(1, 0)`: only addition grouping changes.
First change the recurrence grouping while holding the power grouping fixed;
then change the power accumulation with the same complex bin sums. The two
changes partly cancel.

This is a portable arithmetic witness; compiler builds or candidate geometries
may group the loop differently. The installed-original control below checks dtype, shape and
bytes and remains authoritative on each platform.


In [10]:
import numpy as np
from pycbc.fft import ifft
from pycbc.scheme import CPUScheme, JAXScheme
from pycbc.types import Array
from pycbc.vetoes.chisq import (shift_sum,
                               power_chisq_at_points_from_precomputed)

POINT_DEVICE = 'cpu'  # Repeat with 'cuda:0' on an available NVIDIA GPU.
parts = np.random.default_rng(18).normal(size=(2, 3, 2)).astype(np.float32)
# At sample zero pr=rr=1 and pi=ri=0, so the three products reduce to:
a = parts[:, :, 0].copy()
b = parts[:, :, 1] - a
c = np.zeros_like(a)
per_frequency = np.zeros((2, 2), dtype=np.float32)
for frequency in range(3):
    per_frequency[:, 0] = per_frequency[:, 0] + (a[:, frequency]-c[:, frequency])
    per_frequency[:, 1] = per_frequency[:, 1] + (a[:, frequency]+b[:, frequency])

paired = np.zeros_like(per_frequency)
paired[:, 0] = ((a[:, 1]-c[:, 1])+(a[:, 0]-c[:, 0])) + paired[:, 0]
paired[:, 1] = ((paired[:, 1]+b[:, 0])+(b[:, 1]+a[:, 0])) + a[:, 1]
# The third frequency is the observed native scalar remainder.
paired[:, 0] = (a[:, 2]-c[:, 2]) + paired[:, 0]
paired[:, 1] = (b[:, 2]+paired[:, 1]) + a[:, 2]


def grouped_point_power(sums, interleave_squares=False):
    total = np.float32(0)
    for real, imag in sums:
        real_square, imag_square = real*real, imag*imag
        if interleave_squares:
            total = (total+real_square)+imag_square
        else:
            total = total+(real_square+imag_square)
    return np.asarray([total], dtype=np.float32)


source_order = grouped_point_power(per_frequency)
paired_only = grouped_point_power(paired)
paired_and_total = grouped_point_power(paired, interleave_squares=True)
assert per_frequency.tobytes() != paired.tobytes()
step = np.spacing(source_order).astype(np.float64)
assert ((paired_only.astype(np.float64)-source_order)/step).tolist() == [2.]
assert ((paired_and_total.astype(np.float64)-paired_only)/step).tolist() == [-1.]
print('Per-frequency / paired complex bin sums:', per_frequency, paired)
print('Per-frequency / paired only / paired and total power:',
      source_order, paired_only, paired_and_total)
print('Recurrence / total-regrouping changes in float32 steps: +2 / -1')

# Keep the original full-point assertion tied to the same correlation's SNR.
correlation_values = np.zeros(16, dtype=np.complex64)
correlation_values[1:7] = (parts[:, :, 0]+1j*parts[:, :, 1]).reshape(-1)
edges = np.array([1, 4, 7], dtype=np.uint32)
points = np.array([0], dtype=np.int64)
with CPUScheme():
    correlation = Array(correlation_values.copy())
    transformed = Array(np.zeros_like(correlation_values))
    ifft(correlation, transformed)
    raw_snr = transformed.numpy()[points].copy()
    original_sum = np.asarray(shift_sum(correlation, points, edges)).copy()
    original_point = np.asarray(power_chisq_at_points_from_precomputed(
        correlation, raw_snr.copy(), 1., edges, points)).copy()
with JAXScheme(POINT_DEVICE, reference_operations=('power_chisq_at_points',)):
    selected_point = np.asarray(power_chisq_at_points_from_precomputed(
        Array(correlation_values.copy()), raw_snr.copy(), 1., edges, points))
assert selected_point.dtype == original_point.dtype
assert selected_point.shape == original_point.shape
assert selected_point.tobytes() == original_point.tobytes()
print('Installed original bin power:', original_sum)
print('Same-correlation original point:', original_point)
print('Original point dtype/shape/bytes: exact')


Per-frequency / paired complex bin sums: [[ 2.2554193  -1.3137991 ]
 [ 0.32906145  4.4695587 ]] [[ 2.2554193  -1.3137991 ]
 [ 0.32906145  4.469559  ]]
Per-frequency / paired only / paired and total power: [26.898222] [26.898226] [26.898224]
Recurrence / total-regrouping changes in float32 steps: +2 / -1


Installed original bin power: [26.89822]
Same-correlation original point: [37.15808]
Original point dtype/shape/bytes: exact


## Final SNR scalar precision

Inspiral keeps template normalization in binary64 through `math.sqrt`. Its final multiplication uses a complex64 raw SNR array and a Python scalar; the result remains complex64 on the displayed NumPy version. A normalization change smaller than one float32 step can therefore disappear at this boundary. A larger change can move the scaled components before the final float32 magnitude. This small scalar witness isolates that finish; it does not replay a complete template, PSD or search.


In [16]:
# A binary64 normalization can lose its extra precision at the complex64
# multiplication boundary, before the final float32 magnitude is computed.
raw_snr = np.array([1 + 2j], dtype=np.complex64)
base_norm = float(1.0)
sub_float32_step = float(1.0 + 2.0**-25)
one_float32_step = float(1.0 + 2.0**-23)
base = raw_snr * base_norm
small = raw_snr * sub_float32_step
step = raw_snr * one_float32_step
assert base.dtype == small.dtype == step.dtype == np.complex64
assert np.float32(base_norm).tobytes() == np.float32(sub_float32_step).tobytes()
assert base.tobytes() == small.tobytes()
assert step.tobytes() == (raw_snr * np.float32(one_float32_step)).tobytes()
assert np.abs(step).dtype == np.float32
print('binary64 norms:', base_norm, sub_float32_step, one_float32_step)
print('complex64 scaled SNR:', base, small, step)
print('float32 magnitudes:', np.abs(base), np.abs(small), np.abs(step))


binary64 norms: 1.0 1.0000000298023224 1.0000001192092896
complex64 scaled SNR: [1.+2.j] [1.+2.j] [1.0000001+2.0000002j]
float32 magnitudes: [2.236068] [2.236068] [2.2360682]


## Device reciprocal rounding in PSD division

A componentwise formulation does not guarantee that CPU and GPU quotients have identical rounding. This control freezes the CPU and device float32 reciprocals before multiplying the same complex64 input. On the recorded CUDA implementation, its frozen device reciprocal reproduces the device quotient, while substituting the frozen CPU reciprocal restores the original complex quotient. The printed result reports this numerical fingerprint without identifying a unique instruction on every compiler; other devices may agree throughout. The original `divide` control remains byte-exact.


In [17]:
import jax
try:
    RECIPROCAL_DEVICE = "cuda:0" if jax.devices("gpu") else DEVICE
except RuntimeError:
    RECIPROCAL_DEVICE = DEVICE
with JAXScheme(RECIPROCAL_DEVICE) as reciprocal_context:
    print({"JAX": jax.__version__, "NumPy": np.__version__, "reciprocal platform": reciprocal_context.jax_device.platform, "reciprocal device": reciprocal_context.jax_device.device_kind})
# CPU component division and GPU component division are separate controls.
# Freeze each reciprocal before the product so fusion cannot recreate a
# division and hide which scalar value was used.
from pycbc.types import Array

quotient_input = np.array([complex(-9.398441761732101e-6, 0),
                          complex(-9.350304026156664e-6, 2.3990869522094727e-6)], dtype=np.complex64)
quotient_psd = np.array([87652392.0, 50928592.0], dtype=np.float32)
native_quotient = quotient_input / quotient_psd
cpu_reciprocal = np.float32(1) / quotient_psd
cpu_components = np.empty_like(quotient_input)
cpu_components.real = quotient_input.real / quotient_psd
cpu_components.imag = quotient_input.imag / quotient_psd
assert native_quotient.tobytes() == (quotient_input * cpu_reciprocal).tobytes()
with JAXScheme(RECIPROCAL_DEVICE) as active:
    quotient_gpu = jax.device_put(quotient_input, active.jax_device)
    psd_gpu = jax.device_put(quotient_psd, active.jax_device)
    device_quotient = np.array(jax.jit(lambda x, y: x / y)(quotient_gpu, psd_gpu))
    device_reciprocal = np.array(jax.jit(lambda y: 1.0 / y)(psd_gpu))
    frozen_device_product = np.array(jax.jit(lambda x, r: x * r)(
        quotient_gpu, jax.device_put(device_reciprocal, active.jax_device)))
    frozen_cpu_product = np.array(jax.jit(lambda x, r: x * r)(
        quotient_gpu, jax.device_put(cpu_reciprocal, active.jax_device)))
assert frozen_cpu_product.tobytes() == native_quotient.tobytes()
with JAXScheme(RECIPROCAL_DEVICE, reference_operations=('divide',)) as original_context:
    original_array = Array(quotient_input) / Array(quotient_psd)
    assert original_array.data.device == original_context.jax_device
    original_divide = np.array(original_array.numpy())
assert original_divide.tobytes() == native_quotient.tobytes()
print('CPU and device reciprocals:', cpu_reciprocal, device_reciprocal)
print('CPU complex, CPU components, device quotient:', native_quotient, cpu_components, device_quotient)
print('Frozen device reciprocal reproduces device quotient:', frozen_device_product.tobytes() == device_quotient.tobytes())
print('Frozen CPU reciprocal reproduces original quotient:', frozen_cpu_product.tobytes() == native_quotient.tobytes())


{'JAX': '0.11.2', 'NumPy': '2.2.5', 'reciprocal platform': 'gpu', 'reciprocal device': 'NVIDIA GeForce RTX 3090'}
CPU and device reciprocals: [1.14087015e-08 1.96353351e-08] [1.1408702e-08 1.9635337e-08]
CPU complex, CPU components, device quotient: [-1.0722402e-13+0.0000000e+00j -1.8359636e-13+4.7106877e-14j] [-1.0722402e-13+0.0000000e+00j -1.8359636e-13+4.7106877e-14j] [-1.07224024e-13+0.000000e+00j -1.83596370e-13+4.710688e-14j]
Frozen device reciprocal reproduces device quotient: True
Frozen CPU reciprocal reproduces original quotient: True
